## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-23-10-52-01 +0000,wapo,Who is ready for some presidential debates?,https://www.washingtonpost.com/politics/2026/1...
1,2026-11-23-10-46-56 +0000,bbc,Capita shares plunge and boss steps down as ci...,https://www.bbc.co.uk/news/articles/c65y1qqw3k...
2,2026-11-23-10-45-26 +0000,nyt,Dozens Unaccounted for After Fire Rips Through...,https://www.nytimes.com/2026/11/23/world/asia/...
3,2026-11-23-10-40-43 +0000,nyt,Wall Street Set to Bounce Back After Tough Week,https://www.nytimes.com/2026/11/22/business/ai...
4,2026-11-23-10-36-00 +0000,wsj,How Michigan’s Abdul El-Sayed Helped Ignite a ...,https://www.wsj.com/articles/how-michigans-abd...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2616/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
72,trump,19
560,california,13
50,deal,12
155,new,11
229,russian,10


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
115,2026-11-23-00-00-00 +0000,wsj,Incoming Michigan Democratic Sen.-elect Abdul ...,https://www.wsj.com/politics/abdul-el-sayed-is...,53
204,2026-11-22-15-27-09 +0000,nypost,"Iran taunts Trump, shows off underground ‘miss...",https://nypost.com/2026/11/22/us-news/iran-sho...,50
43,2026-11-23-09-17-10 +0000,nypost,Zelensky confirms Ukraine was behind drone att...,https://nypost.com/2026/11/23/world-news/zelen...,48
164,2026-11-22-19-18-07 +0000,nypost,Trump gives suprising boost to El-Sayed’s heal...,https://nypost.com/2026/11/22/us-news/trump-gi...,48
208,2026-11-22-14-25-59 +0000,wapo,Trump reposts El-Sayed’s ‘TrumpCare’ pitch for...,https://www.washingtonpost.com/politics/2026/1...,47


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
115,53,2026-11-23-00-00-00 +0000,wsj,Incoming Michigan Democratic Sen.-elect Abdul ...,https://www.wsj.com/politics/abdul-el-sayed-is...
204,50,2026-11-22-15-27-09 +0000,nypost,"Iran taunts Trump, shows off underground ‘miss...",https://nypost.com/2026/11/22/us-news/iran-sho...
43,48,2026-11-23-09-17-10 +0000,nypost,Zelensky confirms Ukraine was behind drone att...,https://nypost.com/2026/11/23/world-news/zelen...
148,46,2026-11-22-20-48-02 +0000,chitri,Chuck Hubbard defeats Rep. Virginia Foxx after...,https://www.chicagotribune.com/2026/11/22/chuc...
190,45,2026-11-22-17-11-00 +0000,cbc,COP31 climate talks collapse after member stat...,https://www.cbc.ca/news/world/cop31-talks-coll...
129,39,2026-11-22-22-34-44 +0000,latimes,Missing 19-year-old Marine found dead after we...,https://www.latimes.com/california/story/2026-...
39,37,2026-11-23-09-30-17 +0000,nypost,Jensen Huang’s lover was in contact with Chine...,https://nypost.com/2026/11/23/us-news/jensen-h...
28,36,2026-11-23-10-00-00 +0000,wsj,Democrats making presidential bids will gather...,https://www.wsj.com/politics/elections/democra...
124,30,2026-11-22-23-08-28 +0000,chitri,Automation system problem at Delta causes flig...,https://www.chicagotribune.com/2026/11/22/auto...
233,26,2026-11-22-11-00-00 +0000,wapo,Trump keeps losing in court. Here’s how he get...,https://www.washingtonpost.com/politics/2026/1...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
